In [15]:
import pandas as pd
import numpy as np

df_data = pd.read_excel("dataset_exercicios.xlsx")
display (df_data.head(10))

,ID,Nome,Idade,Salário,Categoria,Notas
0,1,Gabriela,38,NaN,Pleno,2.16
1,2,David,49,1060.0,Junior,13.71
2,3,Eduarda,40,3681.0,Pleno,1.84
3,4,Gabriela,50,4122.0,Pleno,19.30
4,5,Carla,20,4474.0,Junior,9.77
5,6,Henrique,35,4860.0,NaN,3.01
6,7,Eduarda,42,2028.0,Pleno,6.85
7,8,Eduarda,59,2489.0,Sênior,8.27
8,9,Gabriela,48,1603.0,Pleno,15.16
9,10,Bruno,20,1681.0,Pleno,14.87


In [16]:
valores_nulos = df_data.isnull().sum()
display (valores_nulos)

ID            0
Nome          0
Idade         0
Salário      20
Categoria    15
Notas         0
dtype: int64

In [17]:
# 4. Imputação de Valores Omissos (Moda)
from sklearn.impute import SimpleImputer
imputer_mean = SimpleImputer(strategy='most_frequent')
df_data['Salário'] = imputer_mean.fit_transform(df_data[['Salário']])

#print(df_data['Salário'])
print(df_data.isnull().sum())

ID            0
Nome          0
Idade         0
Salário       0
Categoria    15
Notas         0
dtype: int64


In [18]:
# 5. Normalização dos Dados 

from sklearn.preprocessing import MinMaxScaler, StandardScaler

scaler_minmax = MinMaxScaler(feature_range=(0, 1)) 
df_data['Notas_Normalizadas'] = scaler_minmax.fit_transform(df_data[['Notas']])  
print(df_data[['Notas', 'Notas_Normalizadas']].head(10))

scaler_standard = StandardScaler()  
df_data['Salário_Padronizado'] = scaler_standard.fit_transform(df_data[['Salário']])  
print(df_data[['Salário', 'Salário_Padronizado']].head(10))


   Notas  Notas_Normalizadas
0   2.16            0.000025
1  13.71            0.000159
2   1.84            0.000021
3  19.30            0.000224
4   9.77            0.000114
5   3.01            0.000035
6   6.85            0.000080
7   8.27            0.000096
8  15.16            0.000176
9  14.87            0.000173
   Salário  Salário_Padronizado
0   1525.0            -0.044766
1   1060.0            -0.044766
2   3681.0            -0.044766
3   4122.0            -0.044766
4   4474.0            -0.044766
5   4860.0            -0.044766
6   2028.0            -0.044766
7   2489.0            -0.044766
8   1603.0            -0.044766
9   1681.0            -0.044766


In [19]:
# 6. Padronização dos Dados 
df_data['Salário'].fillna(df_data['Salário'].median(), inplace=True)

# Padronizar a coluna 'Salário' utilizando StandardScaler
scaler = StandardScaler()
df_data['Salário_Padronizado'] = scaler.fit_transform(df_data[['Salário']])

# Exibir o resultado
df_data[['ID', 'Salário', 'Salário_Padronizado']]

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18016\2083037468.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_data['Salário'].fillna(df_data['Salário'].median(), inplace=True)


,ID,Salário,Salário_Padronizado
0,1,1525.0,-0.044766
1,2,1060.0,-0.044766
2,3,3681.0,-0.044766
3,4,4122.0,-0.044766
4,5,4474.0,-0.044766
...,...,...,...
495,496,4212.0,-0.044766
496,497,1989.0,-0.044766
497,498,3262.0,-0.044766
498,499,3482.0,-0.044766


In [20]:
# 7. Deteção e Remoção de Outliers  
# 7.1 Z-Score

import numpy as np
from scipy.stats import zscore

# Calcular Z-Score para a coluna 'Salário'
df_data['Z_Score'] = zscore(df_data['Salário'])

# Remover outliers (valores acima de 3 ou abaixo de -3)
df_data_sem_outliers = df_data[np.abs(df_data['Z_Score']) <= 3]

# Exibir os resultados após remoção dos outliers
df_data_sem_outliers[['ID', 'Salário', 'Z_Score']]


,ID,Salário,Z_Score
0,1,1525.0,-0.044766
1,2,1060.0,-0.044766
2,3,3681.0,-0.044766
3,4,4122.0,-0.044766
4,5,4474.0,-0.044766
...,...,...,...
495,496,4212.0,-0.044766
496,497,1989.0,-0.044766
497,498,3262.0,-0.044766
498,499,3482.0,-0.044766


In [21]:
# 7.2. IQR
Q1 = df_data['Notas'].quantile(0.25)
Q3 = df_data['Notas'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

df_data_iqr = df_data[(df_data['Notas'] >= limite_inferior) & (df_data['Notas'] <= limite_superior)]

df_data_iqr[['ID', 'Notas']]


,ID,Notas
0,1,2.16
1,2,13.71
2,3,1.84
3,4,19.30
4,5,9.77
...,...,...
495,496,8.98
496,497,1.69
497,498,7.99
498,499,18.20


In [22]:
#7.3. IsolationForest 

from sklearn.ensemble import IsolationForest

model = IsolationForest(n_estimators=100, contamination='auto')

df_data['Outlier_Salario'] = model.fit_predict(df_data[['Salário']])
df_data['Outlier_Notas'] = model.fit_predict(df_data[['Notas']])

df_data_isolationforest = df_data[(df_data['Outlier_Salario'] == 1) & (df_data['Outlier_Notas'] == 1)]

df_data_isolationforest[['ID', 'Salário', 'Notas', 'Outlier_Salario', 'Outlier_Notas']]


,ID,Salário,Notas,Outlier_Salario,Outlier_Notas
0,1,1525.0,2.16,1,1
2,3,3681.0,1.84,1,1
4,5,4474.0,9.77,1,1
6,7,2028.0,6.85,1,1
7,8,2489.0,8.27,1,1
...,...,...,...,...,...
494,495,2759.0,10.65,1,1
495,496,4212.0,8.98,1,1
496,497,1989.0,1.69,1,1
497,498,3262.0,7.99,1,1


In [23]:
# 7.4. Comparação dos Métodos de Detecção de Outliers 


outliers_zscore = df_data[np.abs(df_data['Z_Score']) > 3]

# IQR
outliers_iqr = df_data[~df_data.index.isin(df_data_iqr.index)]

# IsolationForest
outliers_isolationforest = df_data[~df_data.index.isin(df_data_isolationforest.index)]

# Exibir a contagem de outliers
print(f'Outliers detectados pelo Z-Score: {outliers_zscore.shape[0]}')
print(f'Outliers detectados pelo IQR: {outliers_iqr.shape[0]}')
print(f'Outliers detectados pelo IsolationForest: {outliers_isolationforest.shape[0]}')


Outliers detectados pelo Z-Score: 1
Outliers detectados pelo IQR: 2
Outliers detectados pelo IsolationForest: 141


In [24]:
# 8. Codificação de Variáveis Categóricas 

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

codificador_onehot = OneHotEncoder()

categorias_codificadas = codificador_onehot.fit_transform(df_data[["Categoria"]]).toarray()

df_categorias_codificadas = pd.DataFrame(categorias_codificadas, columns=codificador_onehot.get_feature_names_out(["Categoria"]))

df_final = pd.concat([df_data, df_categorias_codificadas], axis=1)

df_final = df_final.drop("Categoria", axis=1)

print("DataFrame com categorias codificadas:\n", df_final)


DataFrame com categorias codificadas:
       ID      Nome  Idade  Salário  Notas  Notas_Normalizadas  \
0      1  Gabriela     38   1525.0   2.16            0.000025   
1      2     David     49   1060.0  13.71            0.000159   
2      3   Eduarda     40   3681.0   1.84            0.000021   
3      4  Gabriela     50   4122.0  19.30            0.000224   
4      5     Carla     20   4474.0   9.77            0.000114   
..   ...       ...    ...      ...    ...                 ...   
495  496  Gabriela     49   4212.0   8.98            0.000104   
496  497     Carla     20   1989.0   1.69            0.000020   
497  498     David     44   3262.0   7.99            0.000093   
498  499       Ana     46   3482.0  18.20            0.000212   
499  500     David     49   4962.0  16.16            0.000188   

     Salário_Padronizado   Z_Score  Outlier_Salario  Outlier_Notas  \
0              -0.044766 -0.044766                1              1   
1              -0.044766 -0.044766      

In [25]:
# 9. Redução de Dimensionalidade (PCA) 

from sklearn.decomposition import PCA  # Importando o PCA

colunas_pca = ["Idade", "Salário", "Notas"]

modelo_pca = PCA(n_components=2)

modelo_pca.fit(df_final[colunas_pca])

componentes_principais = modelo_pca.transform(df_final[colunas_pca])
df_componentes_principais = pd.DataFrame(componentes_principais, columns=["Componente 1", "Componente 2"])
df_final_pca = pd.concat([df_final, df_componentes_principais], axis=1)

df_final_pca = df_final_pca.drop(colunas_pca, axis=1)

print("DataFrame com PCA:\n", df_final_pca)

DataFrame com PCA:
       ID      Nome  Notas_Normalizadas  Salário_Padronizado   Z_Score  \
0      1  Gabriela            0.000025            -0.044766 -0.044766   
1      2     David            0.000159            -0.044766 -0.044766   
2      3   Eduarda            0.000021            -0.044766 -0.044766   
3      4  Gabriela            0.000224            -0.044766 -0.044766   
4      5     Carla            0.000114            -0.044766 -0.044766   
..   ...       ...                 ...                  ...       ...   
495  496  Gabriela            0.000104            -0.044766 -0.044766   
496  497     Carla            0.000020            -0.044766 -0.044766   
497  498     David            0.000093            -0.044766 -0.044766   
498  499       Ana            0.000212            -0.044766 -0.044766   
499  500     David            0.000188            -0.044766 -0.044766   

     Outlier_Salario  Outlier_Notas  Categoria_Junior  Categoria_Pleno  \
0                  1         